# Aprendizado de Máquina — Lista prática 09

## Métricas para Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O `bank_train_redux.csv` tem 200 variáveis anônimas e cerca de 10% de clientes
positivos. É o cenário em que a acurácia para de funcionar — e onde toda decisão
sobre o **corte** vale mais do que qualquer troca de modelo.

> **um classificador pode ordenar perfeitamente e ainda assim classificar
> *tudo* como negativo. Ordenar e decidir são duas etapas separadas.**

O último exercício investiga uma afirmação da nota de aula que **não se reproduz**
nestes dados — e descobre por quê.

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a acurácia que não diz nada

Carregue o banco. A última coluna do arquivo veio suja da exportação
(`var_199;;;;;;;`), e há `;` no meio de alguns números — a limpeza já está
escrita, mas vale ler: pré-processamento de dado real é assim.

In [ ]:
_nome = "bank_train_redux.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

banco = pd.read_csv(_fonte, nrows=40_000)
banco = banco.replace(to_replace=";", value="", regex=True)
banco = banco.rename(columns={"var_199;;;;;;;": "var_199"})
banco["var_199"] = pd.to_numeric(banco["var_199"])

Xb = banco.drop(columns=["ID_code", "target"]).astype(float).values
yb = banco["target"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    Xb, yb, test_size=0.3, random_state=2026, stratify=...)      # (a) preserve a proporcao

print(f"n = {Xb.shape[0]}, d = {Xb.shape[1]}, prevalencia = {yb.mean():.4f}")
print(f"teste: {len(y_te)} clientes, {int(y_te.sum())} positivos")

In [ ]:
logistica = Pipeline([("escala", StandardScaler()),
                      ("modelo", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

p = logistica.predict_proba(X_te)[:, ...]                         # (a) a coluna da classe positiva

print(f"acuracia do modelo (corte 0,5): {np.mean((p > 0.5).astype(int) == y_te):.4f}")
print(f"acuracia do trivial (sempre 0): {...:.4f}")   # (b)
print(f"quantos o modelo chama de positivo: {int((p > 0.5).sum())} de {len(y_te)}")

---
## Exercício 2 — a matriz de confusão e o corte por custo

Comece pelo corte padrão, depois use a fórmula do Exercício 2 da Lista Teórica 09:

$$p^\ast = \frac{c_{FP}}{c_{FP}+c_{FN}}.$$

In [ ]:
def relatorio(y_real, prob, corte, rotulo):
    vn, fp, fn, vp = confusion_matrix(y_real, (...).astype(int)).ravel()   # (a)
    precisao = vp / (vp + fp)
    recall = ...                                     # (b)
    f1 = 2 * vp / (2 * vp + fp + fn)
    print(f"{rotulo:26s} corte {corte:.4f}  VP {vp:4d}  FP {fp:4d}  FN {fn:4d}"
          f"   precisao {precisao:.4f}  recall {recall:.4f}  F1 {f1:.4f}")
    return fp, fn


fp0, fn0 = relatorio(y_te, p, 0.5, "corte padrao")

In [ ]:
for c_fp, c_fn in [(1, 10), (1, 50)]:
    corte = ...                                # (a)
    fp1, fn1 = relatorio(y_te, p, corte, f"c_FP={c_fp}, c_FN={c_fn}")
    custo_otimo = ...                       # (b)
    custo_padrao = c_fp * fp0 + c_fn * fn0
    print(f"{'':26s} custo {custo_otimo:6d}  contra {custo_padrao:6d} no corte 0,5"
          f"   ({100 * (custo_otimo / custo_padrao - 1):+.0f}%)")

---
## Exercício 3 — as curvas que não dependem do corte

O Exercício 2 mostrou que o corte muda tudo. As curvas ROC e precisão--*recall*
resolvem isso avaliando **todos** os cortes de uma vez.

In [ ]:
auc = ...                                    # (a)
ap = ...                           # (b)

print(f"AUC                 = {auc:.4f}   (acaso = 0,5)")
print(f"average precision   = {ap:.4f}   (acaso = prevalencia = {y_te.mean():.4f})")

In [ ]:
fpr, tpr, _ = roc_curve(y_te, p)
precisao, recall, _ = precision_recall_curve(y_te, p)

fig, (ax1, ax2) = subplots(1, 2, figsize=(9, 3.4))

ax1.plot(fpr, tpr, lw=1.6)
ax1.plot([0, 1], [0, 1], ls="--", lw=1, color="gray")
ax1.set_xlabel("taxa de falsos positivos")
ax1.set_ylabel("recall (taxa de verdadeiros positivos)")
ax1.set_title(f"ROC — AUC = {auc:.4f}")

ax2.plot(recall, precisao, lw=1.6)
ax2.axhline(..., ls="--", lw=1, color="gray")           # (a) a linha de base da PR
ax2.set_xlabel("recall")
ax2.set_ylabel("precisao")
ax2.set_title(f"precisao-recall — AP = {ap:.4f}")

fig.tight_layout()

---
## Exercício 4 — ordenar bem não é estimar bem (às vezes)

A nota de aula afirma que o naive Bayes empata com a logística em AUC e perde
feio em Brier, porque multiplica evidências correlacionadas como se fossem
independentes. Teste a afirmação **neste banco**.

In [ ]:
naive = GaussianNB().fit(X_tr, y_tr)
q = naive.predict_proba(X_te)[:, 1]

for nome, v in (("logistica", p), ("naive Bayes", q)):
    print(f"{nome:12s} AUC {roc_auc_score(y_te, v):.4f}   "
          f"Brier {...:.4f}   "                 # (a)
          f"fracao com p>0,9: {np.mean(v > 0.9):.4f}")
print(f"{'prevalencia':12s} {y_te.mean():.4f}")

A explicação do naive Bayes mal calibrado depende de **as covariáveis serem
correlacionadas**. Meça a correlação neste banco.

In [ ]:
matriz = np.corrcoef(...)                                    # (a) correlacao entre COLUNAS
fora_da_diagonal = matriz[~np.eye(matriz.shape[0], dtype=bool)]

print(f"correlacao absoluta media entre as 200 variaveis: {...:.4f}")   # (b)
print(f"maxima: {np.abs(fora_da_diagonal).max():.4f}")

Para separar causa de coincidência, construa uma população em que você controla a
correlação. Seis covariáveis, 8% de positivos, e um único parâmetro $\rho$
regulando a dependência entre elas.

In [ ]:
def cenario(n, rng, prev=0.08, rho=0.85, d=6):
    y = (rng.uniform(size=n) < prev).astype(int)
    S = np.full((d, d), rho) + np.eye(d) * (1 - rho)
    L = np.linalg.cholesky(S)                       # covariancia S = L L'
    X = rng.normal(size=(n, d)) @ L.T
    X[y == 1] += 1.05                               # as classes diferem so na media
    return X, y


for rho in (..., ...):                                        # (a) e (b)
    X_s, y_s = cenario(4000, np.random.default_rng(2026), rho=rho)
    X_st, y_st = cenario(20000, np.random.default_rng(44), rho=rho)

    print(f"=== rho = {rho} ===")
    for nome, modelo in [("logistica", Pipeline([("e", StandardScaler()),
                                                 ("m", LogisticRegression(max_iter=2000))])),
                         ("naive Bayes", GaussianNB())]:
        v = modelo.fit(X_s, y_s).predict_proba(X_st)[:, 1]
        print(f"  {nome:12s} AUC {roc_auc_score(y_st, v):.4f}   "
              f"Brier {brier_score_loss(y_st, v):.4f}   "
              f"fracao com p>0,9: {np.mean(v > 0.9):.4f}")
    print(f"  prevalencia real: {y_st.mean():.4f}")

> **Sua vez.** Rode o `cenario` com $\rho = 0{,}3$ e $\rho = 0{,}6$. A degradação
> do Brier do naive Bayes é gradual ou tem um limiar?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | acurácia 0,9155 contra 0,9015 do classificador trivial — 1,4 ponto |
| 2 | só mudando o corte, o *recall* vai de 0,28 a 0,96 e o custo cai **79%** |
| 2 | o $F_1$ é máximo num corte que **não** é o de menor custo |
| 3 | AUC 0,8529 e AP 0,4933, contra uma linha de base de 0,0985 |
| 4 | neste banco o naive Bayes **ganha** da logística — o oposto do que a nota prevê |
| 4 | a correlação média entre as 200 variáveis é 0,0048: a hipótese dele é quase exata aqui |
| 4 | com $\rho=0{,}85$ a previsão da nota se confirma: Brier **2,3× pior**, e $p>0{,}9$ para 10,7% dos casos |

**A seguir.** A Aula 10 traz um classificador construído sobre uma ideia
geométrica diferente — a margem — e que, por não estimar probabilidades, deixa a
questão da calibração de fora por construção.